# 4.4 — Programmatic waypoint tasks with TERRA

Build one scene, provide ordered XY waypoints and an arrival radius, generate a terrain-aware reference, and run TERRA-4B using the shared MuscleMimic policy runner.

Run cells from top to bottom. Locally install `pip install -e ".[musclemimic]" mujoco==3.11.0 numpy==2.3.5 scipy==1.16.3 jax==0.10.2 jaxlib==0.10.2 orbax-checkpoint==0.12.7 jupyter`; the Colab copy includes its own installation cell. The procedural reference needs no gated motion dataset or HF_TOKEN. Policy weights are public and pinned.

The reconstructed recipe uses slow walking with terrain-aware ankle/toe IK and explicitly annotated jumps. The original seed-0 demonstration completed all nine goals in 37.93 s. A regenerated reference and physics replay matched that archive exactly. Fresh sampling and library versions can change the rollout; the verification cell reports the actual result.

Credit: [TERRA](https://github.com/amathislab/terra) and [TERRA-4B](https://huggingface.co/merc-s/TERRA-4B) provide the terrain tracking checkpoint; policy execution reuses MyoSuite's MuscleMimic integration.


In [ ]:
import os
import sys
from functools import partial
from itertools import product
from pathlib import Path

if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "osmesa" if "google.colab" in sys.modules else "egl")

import imageio
import matplotlib.pyplot as plt
import mujoco
import numpy as np
from IPython.display import Image, Video, display

from myosuite import make_env
from myosuite.envs.waypoint import WaypointEnv, WaypointTaskCfg
from myosuite.terms.waypoint import WaypointRewardCfg

out_dir = Path("waypoint_outputs")
out_dir.mkdir(exist_ok=True)

## 1. The registered task

`myoFullBodyWaypoint-v0` draws a random 4-waypoint route at every reset (segments of 1–2 m, turns up to ±46°) in front of the MyoFullBody. The **reward** pays the distance closed toward the active waypoint, +1 per waypoint reached and −10 on a fall, so the return is bounded by the route length and ending an episode early never pays. The **observation** is `qpos` without the root's world `x, y`, `qvel · ctrl_dt`, muscle activations and the next two waypoints in the pelvis heading frame. Actions are raw muscle controls.

The goal logic is a set of shared term functions (`myosuite.terms.waypoint`) used by the CPU env and by its mjlab twin, so a policy trained on the GPU replays here.

In [ ]:
env = make_env("myoFullBodyWaypoint-v0")
obs, _ = env.reset(seed=0)
base = env.unwrapped
print("observation", obs.shape, {k: v.shape for k, v in base.get_obs_dict().items()})
print("route (m):", base.waypoints.round(2).tolist())
for _ in range(100):  # zero muscle drive: the body collapses
    obs, reward, terminated, truncated, info = env.step(np.zeros(env.action_space.shape))
    if terminated:
        break
print(f"after {base.data.time:.2f} s: failed={info['failed']}, reward={reward:.2f}")
env.close()

Train the same task on the GPU (mjlab twin, same `env_id`):

```bash
python scripts/train_mjlab.py myoFullBodyWaypoint-v0 --env.scene.num-envs 4096
```

## 2. Build a course programmatically

The scene is a function editing an `MjSpec`: ordinary static box geoms (`size` are half extents). They go in geom group 2 with the floor, where TERRA's heightmap looks for terrain, and use TERRA's terrain friction. Change the dimensions or write any other geometry (rotated boxes, meshes, heightfields) the same way.

In [ ]:
def course_scene(spec: mujoco.MjSpec, rise: float = 0.06, beam_width: float = 0.36, gap: float = 0.20) -> None:
    """Stairs up, a beam with a right-angle turn, a jump, stairs down and a second jump."""

    for geom in spec.geoms:
        geom.group = 0
        if geom.type == mujoco.mjtGeom.mjGEOM_PLANE:
            geom.group = 2
            geom.friction = [0.8, 0.005, 0.0001]

    def box(name: str, x: float, y: float, width: float, length: float, height: float) -> None:
        spec.worldbody.add_geom(
            name=name, type=mujoco.mjtGeom.mjGEOM_BOX, group=2,
            pos=[x, y, height / 2], size=[width / 2, length / 2, height / 2],
            contype=1, conaffinity=1, condim=3, friction=[0.8, 0.005, 0.0001],
            rgba=[0.27, 0.51, 0.71, 1],
        )

    for i in range(4):
        box(f"course_up_{i}", 0, -0.8 - (i + 0.5) * 0.3, 1, 0.3, (i + 1) * rise)
    top = 4 * rise
    box("course_top", 0, -2.3, 0.8, 0.6, top)
    box("course_beam_east", 1.1, -2.3, 1.4, beam_width, top)
    box("course_turn", 2.1, -2.3, 0.6, 0.8, top)
    box("course_beam_south", 2.1, -3.0, beam_width, 0.6, top)
    box("course_jump_landing", 2.1, -3.6 - gap, 0.8, 0.6, top)
    for i in range(3):
        box(f"course_down_{i}", 2.1, -4.05 - gap - i * 0.3, 1, 0.3, (3 - i) * rise)
    box("course_jump2_landing", 2.1, -5.1 - 2 * gap, 0.8, 0.6, rise)


gap = 0.20
waypoints = np.array([
    [0, -1.1], [0, -2.3], [1.05, -2.3], [2.1, -2.3],
    [2.1, -3.1], [2.1, -3.6 - gap], [2.1, -4.7 - gap], [2.1, -5.1 - 2 * gap], [2.1, -6.0 - 2 * gap],
])
route = np.vstack([np.zeros(2), waypoints])
jump_segments = (5, 7)  # Segment indices in route, including its start.


Use the same environment ID for any course; constructor kwargs override its defaults. No additional registration is needed.

```python
from myosuite.core.config import EnvConfig

course = EnvConfig(
    "myoFullBodyWaypoint-v0", max_episode_steps=6000,
    task_kwargs={"edit_fn": course_scene,
                 "task": WaypointTaskCfg(waypoints=tuple(map(tuple, waypoints)), min_site_height=0.5)},
)
env_cpu = make_env(course)
env_gpu = make_env(course, backend="mjlab", num_envs=4096)
```

To load an XML scene instead, supply `model_path` (it replaces the default recipe); use `model_recipe` for another actor recipe.

## 3. The TERRA actor on the course

TERRA-4B was trained on `musclemimic_models` 1.0.5 with two muscle-wrap corrections and the MJX solver settings; `build_terrain_fullbody_spec` rebuilds exactly that actor and applies the scene. Supply this compiled model to `make_env("myoFullBodyWaypoint-v0", model=model)` for CPU playback. For the GPU twin, use a recipe or XML scene through the same environment ID.

In [ ]:
from myosuite.integrations.musclemimic import (
    MIMIC_SITES, load_terrain_policy, TerrainHeights,
    plan_waypoint_reference, GaitParameters, download_terrain_checkpoint, build_terrain_fullbody_spec,
)

model = build_terrain_fullbody_spec(course_scene).compile()
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
heights = TerrainHeights(model, top=10.0)
print(f"TERRA actor: {model.nu} muscles, {model.nq} qpos; ground under the first step: {heights(data, [[0, -0.95]])[0]:.2f} m")

## 4. Generate the reconstructed reference

TERRA tracks a reference; it does not plan a route. `plan_waypoint_reference` queries the scene, holds stance feet fixed, solves ankle/toe IK with the model's joint couplings, and inserts crouch/flight/landing motions for `jump_segments`. It computes forward-difference velocities at 100 Hz and caches kinematics in the existing `MotionClip`.

The successful recipe was obtained through eight iterations: corrected stair heights, settled feet before launch, smoothed seams, and separate foot-tuck heights for the two jumps. Completing the waypoints alone was insufficient: contact checks rejected stepping across a gap. The final recipe uses 0.25 m/s walking, 1.2 s gait cycles, 12 cm foot clearance, 0.45 s reference flight, and 16/10 cm foot tuck.

The historical JAX demonstration used a 10 cm second-jump tuck. This tutorial uses 16/16 cm with the shared NumPy runner. Seeded sampling and numerical differences can affect jumps; changing only the noise sequence did not restore the historical second jump. Both settings remain explicit; `course_verified` requires actual airtime and landing support.

To regenerate the historical reference arrays, set `jump_tuck=0.16, second_jump_tuck=0.10`. Keep the origin spawn, terrain friction and pinned runtime unchanged. This reproduces the reference, while fresh NumPy sampling differs from the historical JAX actions.


In [ ]:
ctrl_dt = 0.01
initial_qpos = model.qpos0.copy()
initial_qpos[:2] = route[0]  # Match the historical spawn, not the MJCF root offset.
parameters = GaitParameters(jump_flight_s=0.45, jump_tuck=0.16, second_jump_tuck=0.16)
reference, planner = plan_waypoint_reference(
    model, route, lambda xy: float(heights(data, [xy])[0]), initial_qpos,
    sites=MIMIC_SITES, jump_segments=jump_segments, parameters=parameters,
)
print(f"Reference: {reference.num_frames} frames, {planner['duration_s']:.2f} s")
print(f"Mean/max ankle+toe IK residual: {planner['mean_foot_ik_error_m']:.4f}/{planner['max_foot_ik_error_m']:.4f} m")
np.savez_compressed(out_dir / "course_reference.npz", qpos=reference.qpos, qvel=reference.qvel)

camera = mujoco.MjvCamera()
camera.lookat[:] = [1, -3.2, 0.3]
camera.distance, camera.azimuth, camera.elevation = 9, 135, -40
data.qpos[:] = reference.qpos[0]
mujoco.mj_forward(model, data)
with mujoco.Renderer(model, 480, 640) as renderer:
    renderer.update_scene(data, camera=camera)
    overview = renderer.render()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(overview)
axes[0].axis("off")
axes[0].set_title("Course and TERRA actor at the reference start")
axes[1].plot(reference.qpos[:, 0], reference.qpos[:, 1], color="gray", label="reference root")
for i, point in enumerate(waypoints):
    axes[1].add_patch(plt.Circle(point, 0.12, color="orange", alpha=0.35))
    axes[1].annotate(str(i + 1), point)
axes[1].set(aspect="equal", xlabel="x (m)", ylabel="y (m)", title="Waypoints (12 cm radius)")
axes[1].legend()
plt.show()

## 5. Run the policy through uninterrupted physics

`load_terrain_policy` restores the pinned checkpoint and frozen normalization using the shared MuscleMimic runner. At each 100 Hz step, call `policy.action_for(data, reference, step)` and pass its muscle controls to `env.step`. The historical demonstration observed the environment after its normal `mj_forward`; this recipe uses the same observation timing directly.

Start from the model's neutral pose and zero velocity. Set `arrival_radius=0.12`; the ordered goals exclude the route's starting point. Reference poses are never written into the controlled simulation after reset. Seed 0 here selects NumPy action sampling; it is not the historical JAX random sequence.


In [ ]:
def support_forces(model, data, ids):
    forces = np.zeros(len(ids))
    for j in range(data.ncon):
        contact = data.contact[j]
        terrain = int(contact.geom1) if int(contact.geom1) in ids else int(contact.geom2)
        if terrain in ids:
            wrench = np.zeros(6)
            mujoco.mj_contactForce(model, data, j, wrench)
            forces[ids[terrain]] += max(0.0, wrench[0])
    return forces


checkpoint = Path(os.environ["TERRA_CHECKPOINT"]) if "TERRA_CHECKPOINT" in os.environ else download_terrain_checkpoint()
policy = load_terrain_policy(checkpoint, model, reference, stochastic=True, seed=0)

task = WaypointTaskCfg(
    site_name="pelvis_mimic", waypoints=tuple(map(tuple, waypoints)),
    arrival_radius=0.12, min_site_height=0.5, reward=WaypointRewardCfg(),
)
env = make_env(
    "myoFullBodyWaypoint-v0", model=model, task=task, initial_qpos=initial_qpos, initial_qvel=np.zeros(model.nv),
    frame_skip=5, render_mode="rgb_array",
)
env.reset(seed=0)
support_names = [model.geom(i).name for i in range(model.ngeom) if model.geom_group[i] == 2]
support_ids = {model.geom(name).id: i for i, name in enumerate(support_names)}
trace = ([], [], [], [])
frames, ret = [], 0.0
render_stride = 20  # Lightweight 5 fps preview; set 4 for a 25 fps video.
with mujoco.Renderer(model, 352, 640) as renderer:
    for step in range(reference.num_frames - 1):
        _, reward, terminated, _, info = env.step(policy.action_for(env.unwrapped.data, reference, step))
        ret += reward
        state = env.unwrapped.data
        trace[0].append(float(state.time))
        trace[1].append(int(info["next_waypoint"]))
        trace[2].append(support_forces(model, state, support_ids))
        trace[3].append([state.sensor(name).data[0] for name in ("r_foot", "r_toes", "l_foot", "l_toes")])
        if step % render_stride == 0:
            follow = mujoco.MjvCamera()
            follow.lookat[:] = env.unwrapped.data.qpos[:3]
            follow.distance, follow.azimuth, follow.elevation = 3.5, 135, -15
            renderer.update_scene(env.unwrapped.data, camera=follow)
            frames.append(renderer.render())
        if terminated:
            break
print(
    f"{env.unwrapped.data.time:.1f} s: {info['next_waypoint']}/{len(waypoints)} waypoints, "
    f"solved={info['solved']}, failed={info['failed']}, return={ret:.2f}"
)
video = out_dir / "terra_course.mp4"
imageio.mimsave(video, frames, fps=1 / (render_stride * ctrl_dt))
preview = out_dir / "terra_course_preview.jpg"
imageio.imwrite(preview, frames[len(frames) // 2], quality=80)
display(Image(filename=str(preview), width=640))
print(f"Video saved to {video}; download it from Colab’s Files panel.")
EMBED_VIDEO = False  # Set True for inline playback; embeds the MP4 in saved outputs.
if EMBED_VIDEO:
    display(Video(str(video), embed=True, width=640))

In [ ]:
def verify_course_trace(trace, support_names, jump_segments, info):
    times, stages, forces, touches = map(np.asarray, trace)
    jumps = []
    pairs = (("course_beam_south", "course_jump_landing"),
             ("course_down_2", "course_jump2_landing"))
    for segment, (source, target) in zip(jump_segments, pairs):
        indices = np.flatnonzero((stages == segment) & (forces.sum(axis=1) < 2)
                                 & (touches.max(axis=1) < 2))
        runs = np.split(indices, np.flatnonzero(np.diff(indices) > 1) + 1)
        candidates = []
        for run in runs:
            if len(run) < 6 or run[0] == 0 or run[-1] + 1 >= len(times):
                continue
            before, after = run[0] - 1, run[-1] + 1
            if (support_names[np.argmax(forces[before])] != source
                    or support_names[np.argmax(forces[after])] != target):
                continue
            landing = (times >= times[after]) & (times <= times[after] + 0.8)
            bilateral = ((touches[landing, :2].sum(axis=1) > 5)
                         & (touches[landing, 2:].sum(axis=1) > 5)).any()
            candidates.append({"airtime_s": round(len(run) * ctrl_dt, 2),
                               "bilateral_support": bool(bilateral)})
        jumps.append(max(candidates, key=lambda item: item["airtime_s"]) if candidates
                     else {"airtime_s": 0.0, "bilateral_support": False})
    floor = support_names.index("floor")
    beam_floor_contacts = int(np.count_nonzero(forces[np.isin(stages, [2, 3, 4]), floor] > 2))
    all_solids = all(forces[:, i].max() > 10 for i, name in enumerate(support_names)
                     if name.startswith("course_"))
    passed = (bool(info["solved"]) and not info["failed"] and all_solids
              and beam_floor_contacts == 0 and len(jumps) == 2
              and all(jump["airtime_s"] >= 0.06 and jump["bilateral_support"] for jump in jumps))
    return {"course_verified": bool(passed), "waypoints": int(info["next_waypoint"]),
            "beam_floor_contacts": beam_floor_contacts, "all_solids_contacted": bool(all_solids),
            "jumps": jumps}

verification = verify_course_trace(trace, support_names, jump_segments, info)
print(verification)
import json
(out_dir / "verification.json").write_text(json.dumps(verification, indent=2))


The printed status describes this physical rollout. A path completion is not automatically a verified jump. Changing the scene requires updating jump annotations and regenerating the reference. The procedural planner is a compact recipe for this demonstration; it does not discover obstacles or guarantee arbitrary-path success.

Validated locally: 9/9 goals with both contact-defined jumps (0.14/0.14 s airtime). In the fresh Colab Python 3.13 runtime, the same 16/16 cm recipe completed 9/9 goals in 38.2 s but stepped across the first gap; only the second jump passed (0.12 s). `course_verified=False` correctly reports that limit. A 20 cm first tuck caused a later fall, and limiting BLAS threads did not fix the gap. These unhelpful changes are not part of this tutorial. Fresh actions are not a platform-independent replay of the archived demonstration.


## 6. Variants

Scene, actor and goal vary independently. Compiling checks MuJoCo structure, not feasibility: re-plan the reference for each one.

In [ ]:
for rise, radius in product((0.04, 0.06), (0.2, 0.3)):
    variant = build_terrain_fullbody_spec(partial(course_scene, rise=rise)).compile()
    trial = WaypointEnv(variant, task=WaypointTaskCfg(waypoints=tuple(map(tuple, waypoints)), arrival_radius=radius))
    observation, _ = trial.reset(seed=0)
    assert trial.observation_space.contains(observation)
    print(f"step rise {rise:.2f} m, arrival radius {radius:.2f} m: {observation.shape[0]} observations")

The default preview is a small JPEG. Set `EMBED_VIDEO = True` for inline playback, or download the MP4 from the Files panel. Before saving a lightweight notebook, use **Edit → Clear all outputs** in Colab.